# tranformer minor project

In [1]:
import requests

url = "https://lwfiles.mycourse.app/62a6cd5e1e9e2fbf212d608d-public/publicFiles/samsum-train.csv"

response = requests.get(url)

with open("samsum-train.csv", "wb") as f:

    f.write(response.content)

In [2]:
url2 = "https://lwfiles.mycourse.app/62a6cd5e1e9e2fbf212d608d-public/publicFiles/samsum-test.csv"

response2 = requests.get(url)

with open("samsum-test.csv", "wb") as f:

    f.write(response2.content)

In [3]:
url3 = "https://lwfiles.mycourse.app/62a6cd5e1e9e2fbf212d608d-public/publicFiles/samsum-validation.csv"

response3 = requests.get(url)

with open("samsum-validation.csv", "wb") as f:

    f.write(response3.content)

In [4]:
%pip install transformers 

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install "transformers[torch]"

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration


In [7]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [8]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [9]:
train_data.shape

(14732, 3)

In [10]:
val_data.shape

(14732, 3)

In [11]:
#we'll take only around 400 samples for both train and val dataset

#random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

## data cleaning
random spaces, special characters , html tags etc are present in the dialogues, therefore we have to remove them and clean the data

In [12]:
import re #import regular expressions module

def clean_data(text):
    text = re.sub(r"\r\n", " ",text) # replace the \r\n\ charcter with " "
    text = re.sub(r"\s+", " ",text) # replace spaces with " "
    text = re.sub(r"<.*?>", " ",text) # replace html tags with " "
    text = text.strip().lower() #remove extra edge spaces and keep all the characters into lower case 
    return text

In [13]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = train_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [14]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

## tokenization
coverting text to numbers

In [15]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [34]:
# coverts raw data to tokens

def tokenize(data):
    inputs = tokenizer(data["dialogue"],padding = "max_length", max_length = 128, truncation = True)
    targets = tokenizer(data["summary"],padding = "max_length", max_length = 32, truncation = True) # max length is less becasue summary is shorter than dialogues

    inputs["labels"] = targets["input_ids"] #token ids => add to input as labesl
    return inputs



In [35]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

#### attention mask
here attention mark is teeling that whihc token has actual meaning and which token is ake and is there for padding purpose

if attention mask = 1 , then the token at that index is real
if attention mask = 0 , then the token at that index is fake 

In [36]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [25208, 1622, 3, 7997, 15, 403, 17, 77, 31, 7, 1108, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

## tuning the model

In [37]:
#  T5ForConditionalGeneration- used to generate text which is based on condition (input)
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [38]:
#fine-tune
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device: ", device)

model.to(device)

device:  mps


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [39]:
training_args = TrainingArguments(
    output_dir="./results",             # Folder where checkpoints, logs and model weights will be saved
    num_train_epochs=6,                 # Number of times the model sees the entire training dataset
    weight_decay=0.01,                  # L2 regularization strength to reduce overfitting
    per_device_train_batch_size=4,      # Number of training samples processed at once on each device (GPU/CPU)
    per_device_eval_batch_size=4,       # Number of validation samples processed at once during evaluation
    eval_strategy="epoch",              # Run validation once after every epoch
    save_strategy="epoch",              # Save model checkpoint once after every epoch
    warmup_steps=100                    # Gradually increase learning rate for first 500 training steps
)

In [40]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [33]:
print(device)
print(next(model.parameters()).device)
print(model.name_or_path)

mps
mps:0
t5-small


In [41]:
# training the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.585853,1.367494
2,1.474734,1.285826
3,1.446435,1.238894
4,1.399433,1.209725
5,1.340832,1.193245
6,1.349097,1.188035


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6000, training_loss=1.4940763244628905, metrics={'train_runtime': 3029.1925, 'train_samples_per_second': 7.923, 'train_steps_per_second': 1.981, 'total_flos': 812050808832000.0, 'train_loss': 1.4940763244628905, 'epoch': 6.0})

In [42]:
#saving the model in the memory, so that it can be reused 

model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [44]:
#reusing the model
model= T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer= T5Tokenizer.from_pretrained("./saved_summary_model")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## testing the model

In [52]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) #clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding = "max_length",
        max_length = 128,
        truncation = True,
        return_tensors = "pt"
    ).to(device)


    # generate the summary --> whihc will be formend in token ids
    model.to(device)
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 64,
        num_beams = 4, #the transformer will give 4 diff outputs , and the best one will be selected 
        early_stopping = True 
    )

    #these token ids have to decoded into eng words 
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # special tokens= spaces , tabs etc
    return summary


    

In [53]:
test_dialogue = """Emma: Hey!! 😊 Are we still meeting tomorrow for the ML project? 🤖
Liam: Yep 👍 Around 2:00 PM at the library 📚
Emma: Great! I'll bring the latest results from the T5 model 📈
Liam: Awesome 😄 I'll prepare the comparison graphs for BERT, T5 and PEGASUS.
Emma: Did you check the validation loss? 🤔
Liam: Yeah, it dropped to 1.2 🎉
Emma: Nice!! That's much better than last week's run 😂
Liam: Definitely. We should also finalize the report before Friday ⚡
Emma: Agreed 👍 See you tomorrow!
Liam: See you 👋"""

summary = summarize_dialogue(test_dialogue)

print("summary: ",summary)

summary:  emma and liam are meeting tomorrow for the ml project. liam will prepare the comparison graphs for bert, t5 and pegasus.
